In [1]:
!pip -q install clickhouse-connect catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 492.7/492.7 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 76.0 MB/s eta 0:00:00


In [2]:
import clickhouse_connect

client = clickhouse_connect.get_client(
    host='of3x1087uw.europe-west4.gcp.clickhouse.cloud',
    port=8443,
    username='default',
    password='dA.k5m79q98DU',
    secure=True
)

client.ping()

True

In [3]:
client.query("""
    SELECT
        split,
        count() AS rows,
        avg(target_purchase_7d) AS target_rate
    FROM mart.ml_dataset_v1
    GROUP BY split
    ORDER BY split
""").result_rows

[('test', 342295, 0.04882046188229451),
 ('train', 987537, 0.04217563493823522),
 ('val', 503018, 0.05790250050694011)]

In [4]:
columns = [
    row[0]
    for row in client.query(
        "DESCRIBE TABLE mart.ml_dataset_v1"
    ).result_rows
]

exclude = {
    'snapshot_id',
    'snapshot_date',
    'snapshot_time',
    'split',
    'clean_user_id',
    'product_id',
    'target_purchase_1d',
    'target_purchase_3d',
    'target_purchase_7d'
}

features = [
    col for col in columns
    if col not in exclude
]

len(features), features

(59,
 ['trigger_event_type',
  'category_id',
  'category_code',
  'brand',
  'price',
  'trigger_multiplier',
  'snapshot_hour',
  'snapshot_day_of_week',
  'snapshot_is_weekend',
  'up_views_today_before',
  'up_carts_today_before',
  'up_cart_multiplier_today_before',
  'up_purchases_today_before',
  'same_session_views_before',
  'same_session_carts_before',
  'same_session_purchases_before',
  'minutes_since_previous_up_event_today',
  'up_views_1d',
  'up_views_7d',
  'up_views_30d',
  'up_carts_1d',
  'up_carts_7d',
  'up_carts_30d',
  'up_cart_multiplier_7d',
  'up_cart_multiplier_30d',
  'up_purchases_7d',
  'up_purchases_30d',
  'up_sessions_7d',
  'up_sessions_30d',
  'hours_since_previous_view',
  'hours_since_previous_cart',
  'hours_since_previous_purchase',
  'user_views_7d',
  'user_views_30d',
  'user_carts_7d',
  'user_carts_30d',
  'user_cart_multiplier_7d',
  'user_cart_multiplier_30d',
  'user_purchases_7d',
  'user_purchases_30d',
  'user_sessions_7d',
  'user_ses

In [5]:
cat_features = [
    'trigger_event_type',
    'category_id',
    'category_code',
    'brand'
]

In [6]:
select_columns = features + ['target_purchase_7d']

sql_columns = ', '.join(select_columns)

train = client.query_df(f"""
    SELECT {sql_columns}
    FROM mart.ml_dataset_v1
    WHERE split = 'train'
""")

val = client.query_df(f"""
    SELECT {sql_columns}
    FROM mart.ml_dataset_v1
    WHERE split = 'val'
""")

In [7]:
print(
    'Train GB:',
    train.memory_usage(deep=True).sum() / 1024**3
)

print(
    'Val GB:',
    val.memory_usage(deep=True).sum() / 1024**3
)

Train GB: 0.5552144171670079
Val GB: 0.2842618618160486


In [8]:
for col in cat_features:
    train[col] = train[col].fillna('__UNKNOWN__').astype(str)
    val[col] = val[col].fillna('__UNKNOWN__').astype(str)

In [9]:
target = 'target_purchase_7d'

X_train = train[features]
y_train = train[target]

X_val = val[features]
y_val = val[target]

In [10]:
print('Train target:', y_train.mean())
print('Val target:', y_val.mean())

Train target: 0.04217563493823522
Val target: 0.05790250050694011


In [11]:
import numpy as np

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score
)

base_rate = y_train.mean()

pred_constant = np.full(
    len(y_val),
    base_rate
)

print(
    'PR-AUC:',
    average_precision_score(y_val, pred_constant)
)

print(
    'ROC-AUC:',
    roc_auc_score(y_val, pred_constant)
)

PR-AUC: 0.05790250050694011
ROC-AUC: 0.5


In [12]:
trigger_rates = (
    train
    .groupby('trigger_event_type')[target]
    .mean()
)

trigger_rates

,target_purchase_7d
trigger_event_type,
cart,0.408571
view,0.025114


In [13]:
pred_trigger = (
    val['trigger_event_type']
    .map(trigger_rates)
    .fillna(base_rate)
)

In [14]:
print(
    'PR-AUC:',
    average_precision_score(y_val, pred_trigger)
)

print(
    'ROC-AUC:',
    roc_auc_score(y_val, pred_trigger)
)

PR-AUC: 0.2728252013689768
ROC-AUC: 0.7179814204673418


In [15]:
from catboost import CatBoostClassifier

model = CatBoostClassifier(
    iterations=1500,
    learning_rate=0.05,
    depth=8,
    loss_function='Logloss',
    eval_metric='AUC',
    l2_leaf_reg=5,
    random_seed=42,
    task_type='GPU',
    devices='0',
    verbose=100,
    allow_writing_files=False
)

In [16]:
model.fit(
    X_train,
    y_train,

    cat_features=cat_features,

    eval_set=(X_val, y_val),

    early_stopping_rounds=100
)

Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.8489488	best: 0.8489488 (0)	total: 125ms	remaining: 3m 7s
100:	test: 0.8785392	best: 0.8785392 (100)	total: 20s	remaining: 4m 37s
200:	test: 0.8814413	best: 0.8814438 (199)	total: 39.5s	remaining: 4m 15s
300:	test: 0.8827640	best: 0.8827640 (300)	total: 52.8s	remaining: 3m 30s
400:	test: 0.8833715	best: 0.8833756 (399)	total: 1m 5s	remaining: 2m 58s
500:	test: 0.8836760	best: 0.8836760 (500)	total: 1m 18s	remaining: 2m 35s
600:	test: 0.8839829	best: 0.8839847 (599)	total: 1m 30s	remaining: 2m 15s
700:	test: 0.8840949	best: 0.8841065 (696)	total: 1m 43s	remaining: 1m 58s
800:	test: 0.8842611	best: 0.8842611 (800)	total: 1m 57s	remaining: 1m 42s
900:	test: 0.8842978	best: 0.8842978 (900)	total: 2m 11s	remaining: 1m 27s
1000:	test: 0.8844649	best: 0.8844649 (1000)	total: 2m 24s	remaining: 1m 12s
1100:	test: 0.8843811	best: 0.8844679 (1009)	total: 2m 37s	remaining: 57.2s
bestTest = 0.8844679296
bestIteration = 1009
Shrink model to first 1010 iterations.


CatBoostClassifier(allow_writing_files=False, depth=8, devices='0', eval_metric='AUC', iterations=1500, l2_leaf_reg=5, learning_rate=0.05, loss_function='Logloss', random_seed=42, task_type='GPU', verbose=100)

In [17]:
pred_val = model.predict_proba(X_val)[:, 1]

In [18]:
print(
    'PR-AUC:',
    average_precision_score(y_val, pred_val)
)

print(
    'ROC-AUC:',
    roc_auc_score(y_val, pred_val)
)

PR-AUC: 0.5066045758794054
ROC-AUC: 0.8844678721097317


In [19]:
mask_view = val['trigger_event_type'] == 'view'

print(
    'VIEW PR-AUC:',
    average_precision_score(
        y_val[mask_view],
        pred_val[mask_view]
    )
)

print(
    'VIEW ROC-AUC:',
    roc_auc_score(
        y_val[mask_view],
        pred_val[mask_view]
    )
)

VIEW PR-AUC: 0.25388978757688363
VIEW ROC-AUC: 0.8124645950931277


In [20]:
mask_cart = val['trigger_event_type'] == 'cart'

print(
    'CART PR-AUC:',
    average_precision_score(
        y_val[mask_cart],
        pred_val[mask_cart]
    )
)

print(
    'CART ROC-AUC:',
    roc_auc_score(
        y_val[mask_cart],
        pred_val[mask_cart]
    )
)

CART PR-AUC: 0.7173593000910156
CART ROC-AUC: 0.6828622333239664


In [21]:
def lift_at_k(y_true, prediction, k=0.10):
    order = np.argsort(-prediction)

    n = max(
        1,
        int(len(order) * k)
    )

    top = order[:n]

    top_rate = y_true.iloc[top].mean()
    base_rate = y_true.mean()

    lift = top_rate / base_rate

    captured = (
        y_true.iloc[top].sum()
        / y_true.sum()
    )

    return {
        'top_rate': top_rate,
        'base_rate': base_rate,
        'lift': lift,
        'captured_positives': captured
    }

In [22]:
lift_at_k(
    y_val.reset_index(drop=True),
    pred_val,
    0.10
)

{'top_rate': np.float64(0.38196059720482695),
 'base_rate': np.float64(0.05790250050694011),
 'lift': np.float64(6.596616620365915),
 'captured_positives': np.float64(0.6596511707752524)}

In [23]:
view_y = y_val[mask_view].reset_index(drop=True)
view_pred = pred_val[mask_view]

lift_at_k(
    view_y,
    view_pred,
    0.10
)

{'top_rate': np.float64(0.16012654249858582),
 'base_rate': np.float64(0.03284008606511869),
 'lift': np.float64(4.87594771161898),
 'captured_positives': np.float64(0.48759170653907496)}

In [24]:
feature_importance = model.get_feature_importance()

importance = list(zip(
    features,
    feature_importance
))

importance = sorted(
    importance,
    key=lambda x: x[1],
    reverse=True
)

for feature, value in importance[:30]:
    print(f'{feature:45s} {value:.4f}')

trigger_event_type                            6.6476
product_purchases_30d                         5.7744
product_purchases_7d                          4.8290
snapshot_hour                                 4.7859
same_session_views_before                     4.3962
user_views_30d                                4.3567
snapshot_day_of_week                          4.1432
user_avg_purchase_price_30d                   3.7819
user_purchases_30d                            3.4804
product_carts_30d                             3.3009
category_id                                   3.1194
product_views_30d                             2.4839
product_carts_7d                              2.3746
hours_since_user_previous_purchase            2.3500
category_code                                 2.2916
up_views_30d                                  2.2289
price_vs_user_purchase_avg                    2.1387
brand                                         2.1377
price                                         

In [25]:
features_no_trigger = [
    col for col in features
    if col != 'trigger_event_type'
]

cat_features_no_trigger = [
    col for col in cat_features
    if col != 'trigger_event_type'
]

In [26]:
model_no_trigger = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.05,
    depth=8,
    loss_function='Logloss',
    eval_metric='AUC',
    l2_leaf_reg=5,
    random_seed=42,
    task_type='GPU',
    devices='0',
    verbose=100,
    allow_writing_files=False
)

model_no_trigger.fit(
    X_train[features_no_trigger],
    y_train,
    cat_features=cat_features_no_trigger,
    eval_set=(
        X_val[features_no_trigger],
        y_val
    ),
    early_stopping_rounds=100
)

Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.8272796	best: 0.8272796 (0)	total: 116ms	remaining: 1m 55s
100:	test: 0.8785062	best: 0.8785062 (100)	total: 13.1s	remaining: 1m 56s
200:	test: 0.8815704	best: 0.8815704 (200)	total: 26.8s	remaining: 1m 46s
300:	test: 0.8823698	best: 0.8823698 (300)	total: 40.4s	remaining: 1m 33s
400:	test: 0.8830316	best: 0.8830316 (400)	total: 53.2s	remaining: 1m 19s
500:	test: 0.8834005	best: 0.8834113 (496)	total: 1m 6s	remaining: 1m 5s
600:	test: 0.8837598	best: 0.8837598 (600)	total: 1m 18s	remaining: 52.2s
700:	test: 0.8838907	best: 0.8839529 (644)	total: 1m 31s	remaining: 39.1s
800:	test: 0.8841229	best: 0.8841268 (795)	total: 1m 44s	remaining: 26s
900:	test: 0.8842641	best: 0.8842648 (897)	total: 1m 57s	remaining: 12.9s
999:	test: 0.8844776	best: 0.8844835 (994)	total: 2m 10s	remaining: 0us
bestTest = 0.8844834566
bestIteration = 994
Shrink model to first 995 iterations.


CatBoostClassifier(allow_writing_files=False, depth=8, devices='0', eval_metric='AUC', iterations=1000, l2_leaf_reg=5, learning_rate=0.05, loss_function='Logloss', random_seed=42, task_type='GPU', verbose=100)

In [27]:
pred_no_trigger = model_no_trigger.predict_proba(
    X_val[features_no_trigger]
)[:, 1]

print(
    'PR-AUC:',
    average_precision_score(
        y_val,
        pred_no_trigger
    )
)

print(
    'ROC-AUC:',
    roc_auc_score(
        y_val,
        pred_no_trigger
    )
)

PR-AUC: 0.5050704969373123
ROC-AUC: 0.8844834448160691


In [28]:
val_meta = client.query_df("""
    SELECT
        snapshot_id,
        snapshot_date
    FROM mart.ml_dataset_v1
    WHERE split = 'val'
""")

Продолжаем

In [29]:
val_check = client.query_df(f"""
    SELECT
        snapshot_id,
        snapshot_date,
        {sql_columns}
    FROM mart.ml_dataset_v1
    WHERE split = 'val'
    ORDER BY snapshot_id
""")

for col in cat_features:
    val_check[col] = (
        val_check[col]
        .fillna('__UNKNOWN__')
        .astype(str)
    )

X_val_check = val_check[features]
y_val_check = val_check[target]

val_check['prediction'] = model.predict_proba(
    X_val_check
)[:, 1]

In [30]:
daily_metrics = []

for date, group in val_check.groupby('snapshot_date'):
    y = group[target]
    p = group['prediction']

    daily_metrics.append({
        'date': date,
        'rows': len(group),
        'target_rate': y.mean(),
        'pr_auc': average_precision_score(y, p),
        'roc_auc': roc_auc_score(y, p)
    })

daily_metrics

[{'date': Timestamp('2019-12-16 00:00:00'),
  'rows': 37365,
  'target_rate': np.float64(0.06270574066639904),
  'pr_auc': np.float64(0.5155472274828234),
  'roc_auc': np.float64(0.8860210616225548)},
 {'date': Timestamp('2019-12-17 00:00:00'),
  'rows': 37699,
  'target_rate': np.float64(0.05347621952836945),
  'pr_auc': np.float64(0.5140214257078085),
  'roc_auc': np.float64(0.8862025286929127)},
 {'date': Timestamp('2019-12-18 00:00:00'),
  'rows': 36086,
  'target_rate': np.float64(0.058360583051598955),
  'pr_auc': np.float64(0.5308323176799584),
  'roc_auc': np.float64(0.8853078971653623)},
 {'date': Timestamp('2019-12-19 00:00:00'),
  'rows': 34207,
  'target_rate': np.float64(0.05715204490309001),
  'pr_auc': np.float64(0.5115475448726599),
  'roc_auc': np.float64(0.893556100567367)},
 {'date': Timestamp('2019-12-20 00:00:00'),
  'rows': 32697,
  'target_rate': np.float64(0.0564577790011316),
  'pr_auc': np.float64(0.508630717583308),
  'roc_auc': np.float64(0.8812454493732202)

In [31]:
import pandas as pd

daily_metrics = pd.DataFrame(daily_metrics)
daily_metrics

,date,rows,target_rate,pr_auc,roc_auc
0,2019-12-16,37365,0.062706,0.515547,0.886021
1,2019-12-17,37699,0.053476,0.514021,0.886203
2,2019-12-18,36086,0.058361,0.530832,0.885308
3,2019-12-19,34207,0.057152,0.511548,0.893556
4,2019-12-20,32697,0.056458,0.508631,0.881245
5,2019-12-21,30697,0.057954,0.491106,0.884264
6,2019-12-22,33404,0.053048,0.495628,0.888886
7,2019-12-23,30777,0.059557,0.529599,0.894067
8,2019-12-24,29164,0.056645,0.517899,0.890668
9,2019-12-25,30665,0.056025,0.518367,0.879231


In [32]:
val_check['score_bin'] = pd.qcut(
    val_check['prediction'],
    10,
    duplicates='drop'
)

calibration = (
    val_check
    .groupby('score_bin', observed=True)
    .agg(
        rows=(target, 'size'),
        avg_prediction=('prediction', 'mean'),
        actual_purchase_rate=(target, 'mean')
    )
    .reset_index()
)

calibration

,score_bin,rows,avg_prediction,actual_purchase_rate
0,"(-0.0009398, 0.00376]",50302,0.002568,0.003201
1,"(0.00376, 0.00562]",50302,0.004690,0.006143
2,"(0.00562, 0.00774]",50302,0.006642,0.008429
3,"(0.00774, 0.0104]",50301,0.009029,0.010656
4,"(0.0104, 0.014]",50302,0.012133,0.013598
5,"(0.014, 0.0192]",50302,0.016447,0.019443
6,"(0.0192, 0.0271]",50301,0.022796,0.027157
7,"(0.0271, 0.0411]",50302,0.033292,0.040674
8,"(0.0411, 0.0905]",50302,0.058261,0.067751
9,"(0.0905, 0.999]",50302,0.345900,0.381973


In [33]:
from sklearn.metrics import brier_score_loss, log_loss

print(
    'Brier:',
    brier_score_loss(
        y_val_check,
        val_check['prediction']
    )
)

print(
    'LogLoss:',
    log_loss(
        y_val_check,
        val_check['prediction']
    )
)

Brier: 0.03810949722056539
LogLoss: 0.1444011467015474


In [34]:
train_view_mask = train['trigger_event_type'] == 'view'
val_view_mask = val['trigger_event_type'] == 'view'

view_features = [
    col for col in features
    if col != 'trigger_event_type'
]

view_cat_features = [
    col for col in cat_features
    if col != 'trigger_event_type'
]

In [35]:
view_model = CatBoostClassifier(
    iterations=1500,
    learning_rate=0.05,
    depth=8,
    loss_function='Logloss',
    eval_metric='AUC',
    l2_leaf_reg=5,
    random_seed=42,
    task_type='GPU',
    devices='0',
    verbose=100,
    allow_writing_files=False
)

view_model.fit(
    X_train.loc[train_view_mask, view_features],
    y_train.loc[train_view_mask],

    cat_features=view_cat_features,

    eval_set=(
        X_val.loc[val_view_mask, view_features],
        y_val.loc[val_view_mask]
    ),

    early_stopping_rounds=100
)

Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.7038780	best: 0.7038780 (0)	total: 140ms	remaining: 3m 29s
100:	test: 0.8036743	best: 0.8036743 (100)	total: 11.9s	remaining: 2m 45s
200:	test: 0.8083966	best: 0.8083966 (200)	total: 23.9s	remaining: 2m 34s
300:	test: 0.8103006	best: 0.8103006 (300)	total: 36s	remaining: 2m 23s
400:	test: 0.8118759	best: 0.8118759 (400)	total: 47.7s	remaining: 2m 10s
500:	test: 0.8130320	best: 0.8130435 (497)	total: 59.5s	remaining: 1m 58s
600:	test: 0.8136663	best: 0.8136663 (600)	total: 1m 11s	remaining: 1m 46s
700:	test: 0.8140615	best: 0.8140647 (699)	total: 1m 23s	remaining: 1m 34s
800:	test: 0.8143751	best: 0.8144257 (780)	total: 1m 35s	remaining: 1m 23s
900:	test: 0.8147217	best: 0.8147217 (900)	total: 1m 47s	remaining: 1m 11s
1000:	test: 0.8150158	best: 0.8150243 (993)	total: 1m 59s	remaining: 59.5s
1100:	test: 0.8151652	best: 0.8151652 (1100)	total: 2m 11s	remaining: 47.7s
1200:	test: 0.8150845	best: 0.8151731 (1101)	total: 2m 23s	remaining: 35.8s
bestTest = 0.8151730895
bestIterati

CatBoostClassifier(allow_writing_files=False, depth=8, devices='0', eval_metric='AUC', iterations=1500, l2_leaf_reg=5, learning_rate=0.05, loss_function='Logloss', random_seed=42, task_type='GPU', verbose=100)

In [36]:
view_pred_special = view_model.predict_proba(
    X_val.loc[val_view_mask, view_features]
)[:, 1]

print(
    'VIEW-only PR-AUC:',
    average_precision_score(
        y_val.loc[val_view_mask],
        view_pred_special
    )
)

print(
    'VIEW-only ROC-AUC:',
    roc_auc_score(
        y_val.loc[val_view_mask],
        view_pred_special
    )
)

VIEW-only PR-AUC: 0.25665928247573555
VIEW-only ROC-AUC: 0.8151730831741211


In [37]:
lift_at_k(
    y_val.loc[val_view_mask].reset_index(drop=True),
    view_pred_special,
    0.10
)

{'top_rate': np.float64(0.16083886782175105),
 'base_rate': np.float64(0.03284008606511869),
 'lift': np.float64(4.897638438060827),
 'captured_positives': np.float64(0.48976076555023923)}

Далее

In [38]:
cal_mask = val_check['snapshot_date'] <= '2019-12-23'
cal_check_mask = val_check['snapshot_date'] > '2019-12-23'

In [39]:
eps = 1e-6

p = val_check['prediction'].clip(eps, 1 - eps)

val_check['raw_logit'] = np.log(
    p / (1 - p)
)

In [40]:
from sklearn.linear_model import LogisticRegression

calibrator = LogisticRegression()

calibrator.fit(
    val_check.loc[cal_mask, ['raw_logit']],
    val_check.loc[cal_mask, target]
)

LogisticRegression()

In [41]:
p_cal_check = calibrator.predict_proba(
    val_check.loc[cal_check_mask, ['raw_logit']]
)[:, 1]

y_cal_check = val_check.loc[
    cal_check_mask,
    target
]

In [42]:
raw_check = val_check.loc[
    cal_check_mask,
    'prediction'
]

print('RAW')
print(
    'Brier:',
    brier_score_loss(
        y_cal_check,
        raw_check
    )
)
print(
    'LogLoss:',
    log_loss(
        y_cal_check,
        raw_check
    )
)

print('\nCALIBRATED')
print(
    'Brier:',
    brier_score_loss(
        y_cal_check,
        p_cal_check
    )
)
print(
    'LogLoss:',
    log_loss(
        y_cal_check,
        p_cal_check
    )
)

RAW
Brier: 0.038563762174065985
LogLoss: 0.14611915070080655

CALIBRATED
Brier: 0.03846083425085199
LogLoss: 0.1457495326538333


Далее

In [43]:
eps = 1e-6

p = val_check['prediction'].clip(eps, 1 - eps)

val_check['raw_logit'] = np.log(
    p / (1 - p)
)

calibrator.fit(
    val_check[['raw_logit']],
    val_check[target]
)

LogisticRegression()

In [44]:
test = client.query_df(f"""
    SELECT
        snapshot_id,
        snapshot_date,
        {sql_columns}
    FROM mart.ml_dataset_v1
    WHERE split = 'test'
    ORDER BY snapshot_id
""")

In [45]:
for col in cat_features:
    test[col] = (
        test[col]
        .fillna('__UNKNOWN__')
        .astype(str)
    )

X_test = test[features]
y_test = test[target]

In [46]:
print(test.shape)
print('Test purchase rate:', y_test.mean())

(342295, 62)
Test purchase rate: 0.04882046188229451


In [47]:
pred_test = model.predict_proba(X_test)[:, 1]

In [48]:
p = np.clip(
    pred_test,
    1e-6,
    1 - 1e-6
)

test_logit = np.log(
    p / (1 - p)
)

test_logit_df = pd.DataFrame({
    'raw_logit': test_logit
})

pred_test_calibrated = calibrator.predict_proba(
    test_logit_df
)[:, 1]

In [49]:
print(
    'TEST PR-AUC:',
    average_precision_score(
        y_test,
        pred_test
    )
)

print(
    'TEST ROC-AUC:',
    roc_auc_score(
        y_test,
        pred_test
    )
)

TEST PR-AUC: 0.5293696083563321
TEST ROC-AUC: 0.9007268789559063


In [50]:
test_view = test['trigger_event_type'] == 'view'

print(
    'TEST VIEW PR-AUC:',
    average_precision_score(
        y_test[test_view],
        pred_test[test_view]
    )
)

print(
    'TEST VIEW ROC-AUC:',
    roc_auc_score(
        y_test[test_view],
        pred_test[test_view]
    )
)

TEST VIEW PR-AUC: 0.2892258266138703
TEST VIEW ROC-AUC: 0.8377272465689211


In [51]:
test_cart = test['trigger_event_type'] == 'cart'

print(
    'TEST CART PR-AUC:',
    average_precision_score(
        y_test[test_cart],
        pred_test[test_cart]
    )
)

print(
    'TEST CART ROC-AUC:',
    roc_auc_score(
        y_test[test_cart],
        pred_test[test_cart]
    )
)

TEST CART PR-AUC: 0.7348236490825323
TEST CART ROC-AUC: 0.7183101907155282


In [52]:
print(
    'TEST Lift@10 overall:',
    lift_at_k(
        y_test.reset_index(drop=True),
        pred_test,
        0.10
    )
)

TEST Lift@10 overall: {'top_rate': np.float64(0.3436559642408484), 'base_rate': np.float64(0.04882046188229451), 'lift': np.float64(7.039178880965903), 'captured_positives': np.float64(0.7039076057686554)}


In [53]:
print(
    'TEST Lift@10 VIEW:',
    lift_at_k(
        y_test[test_view].reset_index(drop=True),
        pred_test[test_view],
        0.10
    )
)

TEST Lift@10 VIEW: {'top_rate': np.float64(0.14861021924594073), 'base_rate': np.float64(0.027376693269730606), 'lift': np.float64(5.428348039763208), 'captured_positives': np.float64(0.5428348039763208)}


In [54]:
print(
    'TEST Lift@5 overall:',
    lift_at_k(
        y_test.reset_index(drop=True),
        pred_test,
        0.05
    )
)

print(
    'TEST Lift@5 VIEW:',
    lift_at_k(
        y_test[test_view].reset_index(drop=True),
        pred_test[test_view],
        0.05
    )
)

TEST Lift@5 overall: {'top_rate': np.float64(0.5375715788243544), 'base_rate': np.float64(0.04882046188229451), 'lift': np.float64(11.01119403827912), 'captured_positives': np.float64(0.5505355753695171)}
TEST Lift@5 VIEW: {'top_rate': np.float64(0.2323405296312152), 'base_rate': np.float64(0.027376693269730606), 'lift': np.float64(8.486800335674781), 'captured_positives': np.float64(0.4243270412152351)}


In [55]:
print('RAW')

print(
    'Brier:',
    brier_score_loss(
        y_test,
        pred_test
    )
)

print(
    'LogLoss:',
    log_loss(
        y_test,
        pred_test
    )
)

print('\nCALIBRATED')

print(
    'Brier:',
    brier_score_loss(
        y_test,
        pred_test_calibrated
    )
)

print(
    'LogLoss:',
    log_loss(
        y_test,
        pred_test_calibrated
    )
)

RAW
Brier: 0.031136424219942307
LogLoss: 0.11989958127694902

CALIBRATED
Brier: 0.031098917689961574
LogLoss: 0.12024666551902127


In [56]:
print(
    'Actual purchase rate:',
    y_test.mean()
)

print(
    'Mean raw probability:',
    pred_test.mean()
)

print(
    'Mean calibrated probability:',
    pred_test_calibrated.mean()
)

Actual purchase rate: 0.04882046188229451
Mean raw probability: 0.04728210964835377
Mean calibrated probability: 0.053526413453598036


Почти конец

In [57]:
from catboost import Pool

shap_sample = test.sample(
    n=20000,
    random_state=42
).copy()

X_shap = shap_sample[features]

shap_pool = Pool(
    X_shap,
    cat_features=cat_features
)

shap_values = model.get_feature_importance(
    shap_pool,
    type='ShapValues'
)

In [58]:
shap_contrib = shap_values[:, :-1]

In [59]:
shap_importance = []

for i, feature in enumerate(features):
    value = abs(shap_contrib[:, i]).mean()

    shap_importance.append(
        (feature, value)
    )

shap_importance.sort(
    key=lambda x: x[1],
    reverse=True
)

for feature, value in shap_importance[:30]:
    print(
        f'{feature:45s} {value:.6f}'
    )

product_purchases_30d                         0.196382
snapshot_hour                                 0.190051
user_views_30d                                0.185703
category_id                                   0.158457
trigger_event_type                            0.155082
product_purchases_7d                          0.144301
product_views_30d                             0.140198
brand                                         0.140163
product_carts_30d                             0.135266
product_views_7d                              0.109921
user_purchases_30d                            0.109329
snapshot_day_of_week                          0.108356
product_carts_7d                              0.103085
user_avg_purchase_price_30d                   0.092672
snapshot_is_weekend                           0.091826
price                                         0.079048
user_sessions_30d                             0.072426
product_cart_multiplier_30d                   0.063987
hours_sinc

Уже скоро

In [60]:
import pandas as pd
import numpy as np


shap_df = pd.DataFrame(
    shap_contrib,
    columns=features,
    index=X_shap.index
)

In [61]:
def shap_by_bins(feature, bins=10):
    df = pd.DataFrame({
        'value': X_shap[feature],
        'shap': shap_df[feature]
    }).dropna()

    df['bin'] = pd.qcut(
        df['value'],
        bins,
        duplicates='drop'
    )

    result = (
        df
        .groupby('bin', observed=True)
        .agg(
            rows=('value', 'size'),
            avg_value=('value', 'mean'),
            avg_shap=('shap', 'mean')
        )
        .reset_index()
    )

    return result

In [62]:
shap_by_bins(
    'product_purchases_30d'
)

,bin,rows,avg_value,avg_shap
0,"(-0.001, 1.0]",4015,0.280199,-0.255899
1,"(1.0, 5.0]",2174,3.172033,-0.178990
2,"(5.0, 13.0]",1981,8.905098,-0.088163
3,"(13.0, 31.0]",1858,21.259957,-0.008115
4,"(31.0, 86.0]",1986,53.458207,0.031961
5,"(86.0, 282.0]",1987,163.425767,0.092568
6,"(282.0, 1191.2]",1999,613.950475,0.313958
7,"(1191.2, 5799.0]",2000,2803.330500,0.317816
8,"(5799.0, 35395.0]",2000,17588.865000,0.393209


In [63]:
shap_by_bins(
    'user_views_30d'
)

,bin,rows,avg_value,avg_shap
0,"(-0.001, 2.0]",6235,0.333119,0.198389
1,"(2.0, 6.0]",1798,4.351502,0.183532
2,"(6.0, 14.0]",2173,10.197423,0.133193
3,"(14.0, 24.0]",1819,19.186366,0.021317
4,"(24.0, 41.0]",2016,32.243056,-0.048609
5,"(41.0, 70.0]",1979,54.629611,-0.111218
6,"(70.0, 144.0]",2003,100.279081,-0.240291
7,"(144.0, 1785.0]",1977,339.554881,-0.492649


In [64]:
shap_by_bins(
    'hours_since_previous_view'
)

,bin,rows,avg_value,avg_shap
0,"(4.0489999999999995, 18.143]",312,13.574768,0.073841
1,"(18.143, 25.435]",311,22.058308,0.101722
2,"(25.435, 41.681]",311,32.450512,0.116100
3,"(41.681, 64.912]",312,51.101782,0.127639
4,"(64.912, 97.135]",311,79.350376,0.122749
5,"(97.135, 145.915]",311,121.494029,0.130590
6,"(145.915, 216.981]",312,181.528906,0.136826
7,"(216.981, 330.113]",311,269.055230,0.140659
8,"(330.113, 497.775]",311,407.918161,0.134958
9,"(497.775, 732.366]",312,598.876627,0.126792


In [65]:
shap_by_bins(
    'hours_since_user_previous_purchase'
)

,bin,rows,avg_value,avg_shap
0,"(4.614999999999999, 25.827]",370,18.659182,0.108907
1,"(25.827, 48.902]",369,36.732468,0.147283
2,"(48.902, 93.111]",369,69.094573,0.156869
3,"(93.111, 144.216]",369,116.544088,0.163361
4,"(144.216, 241.078]",369,191.682132,0.198519
5,"(241.078, 327.895]",369,284.909440,0.207653
6,"(327.895, 433.588]",369,382.630836,0.185787
7,"(433.588, 524.412]",369,477.357586,0.173431
8,"(524.412, 619.869]",369,566.969602,0.178246
9,"(619.869, 738.301]",369,674.911869,0.176324


In [66]:
shap_by_bins(
    'same_session_views_before'
)

,bin,rows,avg_value,avg_shap
0,"(-0.001, 17.0]",20000,0.07315,0.005293


In [67]:
hour_shap = pd.DataFrame({
    'hour': X_shap['snapshot_hour'],
    'shap': shap_df['snapshot_hour']
})

hour_result = (
    hour_shap
    .groupby('hour')
    .agg(
        rows=('shap', 'size'),
        avg_shap=('shap', 'mean')
    )
    .reset_index()
)

hour_result

,hour,rows,avg_shap
0,0,138,-0.190163
1,1,153,-0.211616
2,2,389,-0.193517
3,3,563,-0.026430
4,4,811,0.128998
5,5,955,0.146257
6,6,1062,0.165609
7,7,1144,0.190027
8,8,1203,0.221053
9,9,1150,0.197092


In [68]:
brand_shap = pd.DataFrame({
    'brand': X_shap['brand'],
    'shap': shap_df['brand']
})

brand_result = (
    brand_shap
    .groupby('brand')
    .agg(
        rows=('shap', 'size'),
        avg_shap=('shap', 'mean')
    )
    .query('rows >= 100')
    .sort_values(
        'avg_shap',
        ascending=False
    )
)

brand_result.head(20)

,rows,avg_shap
brand,,
apple,1975,0.145830
samsung,2534,0.142325
huawei,543,0.066060
oppo,181,0.034685
xiaomi,1296,0.013136
artel,153,-0.028927
lenovo,192,-0.049776
sony,256,-0.053331
acer,174,-0.054388


In [77]:
print(shap_by_bins('product_purchases_30d'), '\n',
shap_by_bins('user_views_30d'), '\n',
shap_by_bins('user_purchases_30d'), '\n',
shap_by_bins('hours_since_user_previous_purchase'), '\n',
shap_by_bins('up_views_30d'), '\n',
shap_by_bins('same_session_views_before')
)

                 bin  rows     avg_value  avg_shap
0      (-0.001, 1.0]  4015      0.280199 -0.255899
1         (1.0, 5.0]  2174      3.172033 -0.178990
2        (5.0, 13.0]  1981      8.905098 -0.088163
3       (13.0, 31.0]  1858     21.259957 -0.008115
4       (31.0, 86.0]  1986     53.458207  0.031961
5      (86.0, 282.0]  1987    163.425767  0.092568
6    (282.0, 1191.2]  1999    613.950475  0.313958
7   (1191.2, 5799.0]  2000   2803.330500  0.317816
8  (5799.0, 35395.0]  2000  17588.865000  0.393209 
                bin  rows   avg_value  avg_shap
0    (-0.001, 2.0]  6235    0.333119  0.198389
1       (2.0, 6.0]  1798    4.351502  0.183532
2      (6.0, 14.0]  2173   10.197423  0.133193
3     (14.0, 24.0]  1819   19.186366  0.021317
4     (24.0, 41.0]  2016   32.243056 -0.048609
5     (41.0, 70.0]  1979   54.629611 -0.111218
6    (70.0, 144.0]  2003  100.279081 -0.240291
7  (144.0, 1785.0]  1977  339.554881 -0.492649 
              bin   rows  avg_value  avg_shap
0  (-0.001, 1.0]  

In [78]:
hour_result

,hour,rows,avg_shap
0,0,138,-0.190163
1,1,153,-0.211616
2,2,389,-0.193517
3,3,563,-0.026430
4,4,811,0.128998
5,5,955,0.146257
6,6,1062,0.165609
7,7,1144,0.190027
8,8,1203,0.221053
9,9,1150,0.197092


Иииии

In [79]:
tmp = pd.DataFrame({
    'views': X_shap['same_session_views_before'],
    'shap': shap_df['same_session_views_before']
})

tmp['group'] = pd.cut(
    tmp['views'],
    bins=[-1, 0, 1, 2, 3, float('inf')],
    labels=['0', '1', '2', '3', '4+']
)

same_session_result = (
    tmp
    .groupby('group', observed=True)
    .agg(
        rows=('shap', 'size'),
        avg_shap=('shap', 'mean')
    )
    .reset_index()
)

same_session_result

,group,rows,avg_shap
0,0,19073,-0.028073
1,1,635,0.680901
2,2,165,0.714463
3,3,71,0.714942
4,4+,56,0.719182


In [80]:
tmp = pd.DataFrame({
    'purchases': X_shap['user_purchases_30d'],
    'shap': shap_df['user_purchases_30d']
})

tmp['group'] = pd.cut(
    tmp['purchases'],
    bins=[-1, 0, 1, 2, 3, 5, 10, float('inf')],
    labels=['0', '1', '2', '3', '4–5', '6–10', '11+']
)

(
    tmp
    .groupby('group', observed=True)
    .agg(
        rows=('shap', 'size'),
        avg_shap=('shap', 'mean')
    )
)

,rows,avg_shap
group,,
0,16309,-0.047894
1,1732,-0.035163
2,632,0.156801
3,328,0.389323
4–5,301,0.838388
6–10,342,1.058753
11+,356,1.407539


In [81]:
hour_actual = (
    test
    .groupby('snapshot_hour')
    .agg(
        rows=(target, 'size'),
        purchase_rate=(target, 'mean')
    )
    .reset_index()
)

hour_actual

,snapshot_hour,rows,purchase_rate
0,0,2229,0.039928
1,1,2569,0.040872
2,2,6657,0.035602
3,3,9985,0.059089
4,4,13585,0.060802
5,5,16234,0.066342
6,6,17973,0.066656
7,7,19161,0.064193
8,8,19946,0.065126
9,9,19177,0.059603


Почти конец

In [82]:
def gains_curve(y_true, prediction, steps=100):
    df = pd.DataFrame({
        'y': np.asarray(y_true),
        'pred': prediction
    })

    df = df.sort_values(
        'pred',
        ascending=False
    ).reset_index(drop=True)

    total_positive = df['y'].sum()
    base_rate = df['y'].mean()

    result = []

    for i in range(1, steps + 1):
        fraction = i / steps
        n = max(1, int(len(df) * fraction))

        top = df.iloc[:n]

        captured = top['y'].sum() / total_positive
        top_rate = top['y'].mean()
        lift = top_rate / base_rate

        result.append({
            'top_pct': fraction * 100,
            'captured_purchase_pct': captured * 100,
            'purchase_rate_pct': top_rate * 100,
            'lift': lift
        })

    return pd.DataFrame(result)

In [83]:
gains_test = gains_curve(
    y_test,
    pred_test
)

gains_test.head(10)

,top_pct,captured_purchase_pct,purchase_rate_pct,lift
0,1.0,16.881096,82.437171,16.885783
1,2.0,29.567351,72.184076,14.785619
2,3.0,39.764227,64.715621,13.255840
3,4.0,48.261624,58.907311,12.066111
4,5.0,55.053558,53.757158,11.011194
5,6.0,59.954521,48.785120,9.992761
6,7.0,63.927952,44.586811,9.132812
7,8.0,66.734486,40.725998,8.341994
8,9.0,68.649393,37.239499,7.627846
9,10.0,70.390761,34.365596,7.039179


In [84]:
gains_view_test = gains_curve(
    y_test[test_view],
    pred_test[test_view]
)

gains_view_test.head(10)

,top_pct,captured_purchase_pct,purchase_rate_pct,lift
0,1.0,19.792248,54.189602,19.794064
1,2.0,28.716631,39.311927,14.359633
2,3.0,34.792807,31.753313,11.598666
3,4.0,39.137719,26.786943,9.784579
4,5.0,42.432704,23.234053,8.486800
5,6.0,45.314420,20.676826,7.552711
6,7.0,47.872222,18.722698,6.838919
7,8.0,50.161957,17.166119,6.270340
8,9.0,52.284151,15.904458,5.809488
9,10.0,54.283480,14.861022,5.428348


Графики

In [85]:
gains_test_export = gains_test.copy()
gains_test_export['segment'] = 'Overall'

gains_view_export = gains_view_test.copy()
gains_view_export['segment'] = 'View only'

gains_export = pd.concat(
    [
        gains_test_export,
        gains_view_export
    ],
    ignore_index=True
)

gains_export.head()

,top_pct,captured_purchase_pct,purchase_rate_pct,lift,segment
0,1.0,16.881096,82.437171,16.885783,Overall
1,2.0,29.567351,72.184076,14.785619,Overall
2,3.0,39.764227,64.715621,13.255840,Overall
3,4.0,48.261624,58.907311,12.066111,Overall
4,5.0,55.053558,53.757158,11.011194,Overall


In [88]:
random_curve = pd.DataFrame({
    'top_pct': np.arange(1, 101, dtype=float),
    'captured_purchase_pct': np.arange(1, 101, dtype=float),
    'purchase_rate_pct': np.full(100, np.nan, dtype=float),
    'lift': np.ones(100, dtype=float),
    'segment': ['Random'] * 100
})

gains_export = pd.concat(
    [
        gains_test_export,
        gains_view_export,
        random_curve
    ],
    ignore_index=True
)

In [87]:
gains_export.to_csv(
    'purchase_gains_curve.csv',
    index=False
)

Проводим test этой же модели на 5% всех данных без пересечения с уже использованными

In [89]:
holdout5 = client.query_df(f"""
    SELECT
        snapshot_id,
        snapshot_date,
        {sql_columns}
    FROM mart.ml_dataset_holdout5_v1
    ORDER BY snapshot_id
""")

In [90]:
for col in cat_features:
    holdout5[col] = (
        holdout5[col]
        .fillna('__UNKNOWN__')
        .astype(str)
    )

In [91]:
X_holdout5 = holdout5[features]
y_holdout5 = holdout5[target]

print(holdout5.shape)
print('Purchase rate:', y_holdout5.mean())

(857588, 62)
Purchase rate: 0.049207778093909894


In [92]:
pred_holdout5 = model.predict_proba(
    X_holdout5
)[:, 1]

In [93]:
print(
    'HOLDOUT-5 PR-AUC:',
    average_precision_score(
        y_holdout5,
        pred_holdout5
    )
)

print(
    'HOLDOUT-5 ROC-AUC:',
    roc_auc_score(
        y_holdout5,
        pred_holdout5
    )
)

HOLDOUT-5 PR-AUC: 0.5348767951122684
HOLDOUT-5 ROC-AUC: 0.9025872464879743


In [94]:
mask_view_5 = (
    holdout5['trigger_event_type'] == 'view'
)

print(
    'HOLDOUT-5 VIEW PR-AUC:',
    average_precision_score(
        y_holdout5[mask_view_5],
        pred_holdout5[mask_view_5]
    )
)

print(
    'HOLDOUT-5 VIEW ROC-AUC:',
    roc_auc_score(
        y_holdout5[mask_view_5],
        pred_holdout5[mask_view_5]
    )
)

HOLDOUT-5 VIEW PR-AUC: 0.2875757029480436
HOLDOUT-5 VIEW ROC-AUC: 0.8408073061833624


In [95]:
mask_cart_5 = (
    holdout5['trigger_event_type'] == 'cart'
)

print(
    'HOLDOUT-5 CART PR-AUC:',
    average_precision_score(
        y_holdout5[mask_cart_5],
        pred_holdout5[mask_cart_5]
    )
)

print(
    'HOLDOUT-5 CART ROC-AUC:',
    roc_auc_score(
        y_holdout5[mask_cart_5],
        pred_holdout5[mask_cart_5]
    )
)

HOLDOUT-5 CART PR-AUC: 0.7458933102994871
HOLDOUT-5 CART ROC-AUC: 0.7194277053936433


In [96]:
print(
    'HOLDOUT-5 Lift@5 overall:',
    lift_at_k(
        y_holdout5.reset_index(drop=True),
        pred_holdout5,
        0.05
    )
)

print(
    'HOLDOUT-5 Lift@10 overall:',
    lift_at_k(
        y_holdout5.reset_index(drop=True),
        pred_holdout5,
        0.10
    )
)

print(
    'HOLDOUT-5 Lift@5 VIEW:',
    lift_at_k(
        y_holdout5[mask_view_5].reset_index(drop=True),
        pred_holdout5[mask_view_5],
        0.05
    )
)

print(
    'HOLDOUT-5 Lift@10 VIEW:',
    lift_at_k(
        y_holdout5[mask_view_5].reset_index(drop=True),
        pred_holdout5[mask_view_5],
        0.10
    )
)

HOLDOUT-5 Lift@5 overall: {'top_rate': np.float64(0.5386086429254413), 'base_rate': np.float64(0.049207778093909894), 'lift': np.float64(10.945599736235625), 'captured_positives': np.float64(0.5472748815165877)}
HOLDOUT-5 Lift@10 overall: {'top_rate': np.float64(0.3483989831852422), 'base_rate': np.float64(0.049207778093909894), 'lift': np.float64(7.080160833930462), 'captured_positives': np.float64(0.7080094786729858)}
HOLDOUT-5 Lift@5 VIEW: {'top_rate': np.float64(0.23740849194729136), 'base_rate': np.float64(0.027567038635825682), 'lift': np.float64(8.612041905682212), 'captured_positives': np.float64(0.4306010445250952)}
HOLDOUT-5 Lift@10 VIEW: {'top_rate': np.float64(0.15100048804294777), 'base_rate': np.float64(0.027567038635825682), 'lift': np.float64(5.47757378069293), 'captured_positives': np.float64(0.5477560414269275)}


Сохраняем модель

In [97]:
import pickle

model.save_model("purchase_propensity_v1.cbm")

metadata = {
    "features": features,
    "cat_features": cat_features,
    "target": target,
    "prediction_horizon_days": 7,
    "history_window_days": 30
}

with open("purchase_propensity_metadata.pkl", "wb") as f:
    pickle.dump(metadata, f)

In [ ]:
# Как воспользоваться моделью

# from catboost import CatBoostClassifier

# model = CatBoostClassifier()
# model.load_model("purchase_propensity_v1.cbm")

# pred = model.predict_proba(X_new)[:, 1]